In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from datetime import datetime
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

# =============================================================================
# SETUP & CONFIGURATION
# =============================================================================

# Import configuration
from config.config import config
paths = config['paths']
model_config = config['model']
feature_config = config['features']

# Create directories
for d in [paths.bronze_dir, paths.silver_dir, paths.features_dir, 
          paths.models_dir, paths.results_dir, paths.eda_dir]:
    os.makedirs(d, exist_ok=True)
print("=" * 80)
print("BEHAVIORAL CREDIT RISK SCORING SYSTEM")
print(f"Started at: {datetime.now()}")
print("=" * 80)
print(f"Data Directory: {paths.data_dir}")
print(f"Bronze Directory: {paths.bronze_dir}")
print(f"Silver Directory: {paths.silver_dir}")
print(f"Features Directory: {paths.features_dir}")
print(f"Models Directory: {paths.models_dir}")
print(f"Results Directory: {paths.results_dir}")
print("=" * 80)

BEHAVIORAL CREDIT RISK SCORING SYSTEM
Started at: 2026-07-10 17:25:59.089165
Data Directory: D:/Projects/credit_risk_scoring/data
Bronze Directory: D:/Projects/credit_risk_scoring/data/bronze
Silver Directory: D:/Projects/credit_risk_scoring/data/silver
Features Directory: D:/Projects/credit_risk_scoring/data/features
Models Directory: D:/Projects/credit_risk_scoring/models
Results Directory: D:/Projects/credit_risk_scoring/results


In [3]:

# =============================================================================
# SPARK SESSION CREATION
# =============================================================================

from data_ingestion.data_ingestion import SFLLDDataIngestion, create_spark_session

# Create Spark session
spark = create_spark_session()
print(f"Spark Session created: {spark.version}")

# Initialize components
ingestor = SFLLDDataIngestion(spark)

# =============================================================================
# MODULE 1: DATA INGESTION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 1: DATA INGESTION")
print("=" * 80)


fs.defaultFS = file:///
fs.default.name = file:///
Spark Session created: 3.5.1

MODULE 1: DATA INGESTION


In [4]:
spark

In [6]:

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING BRONZE DATA (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING DATA - Use this if you already have bronze data
print("\nLoading existing bronze data...")
origination_df = spark.read.parquet(paths.origination_bronze)
performance_df = spark.read.parquet(paths.performance_bronze)
print(f"Origination: {origination_df.count():,} loans")
print(f"Performance: {performance_df.count():,} records")


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Use this for first time or to re-ingest
print("\nIngesting data from scratch...")

raw_dir = paths.raw_dir
years = list(range(1999, 2013))

# Ingest origination
print("Ingesting origination data...")
origination_df = ingestor.ingest_all_years(
    raw_dir=raw_dir,
    years=years,
    bronze_dir=paths.bronze_dir,
    file_prefix="sample",
    data_type="origination"
)

# Ingest performance
print("Ingesting performance data...")
performance_df = ingestor.ingest_all_years(
    raw_dir=raw_dir,
    years=years,
    bronze_dir=paths.bronze_dir,
    file_prefix="sample",
    data_type="performance"
)

print(f"Origination: {origination_df.count():,} loans")
print(f"Performance: {performance_df.count():,} records")
"""


Loading existing bronze data...
Origination: 700,000 loans
Performance: 43,612,276 records


'\n# PROCESS FROM SCRATCH - Use this for first time or to re-ingest\nprint("\nIngesting data from scratch...")\n\nraw_dir = paths.raw_dir\nyears = list(range(1999, 2013))\n\n# Ingest origination\nprint("Ingesting origination data...")\norigination_df = ingestor.ingest_all_years(\n    raw_dir=raw_dir,\n    years=years,\n    bronze_dir=paths.bronze_dir,\n    file_prefix="sample",\n    data_type="origination"\n)\n\n# Ingest performance\nprint("Ingesting performance data...")\nperformance_df = ingestor.ingest_all_years(\n    raw_dir=raw_dir,\n    years=years,\n    bronze_dir=paths.bronze_dir,\n    file_prefix="sample",\n    data_type="performance"\n)\n\nprint(f"Origination: {origination_df.count():,} loans")\nprint(f"Performance: {performance_df.count():,} records")\n'

In [7]:

# =============================================================================
# MODULE 2: DATA CLEANING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 2: DATA CLEANING")
print("=" * 80)

from preprocessing.cleaning import SFLLDDataCleaner

# Initialize cleaner
cleaner = SFLLDDataCleaner(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING SILVER DATA (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING CLEANED DATA
print("\nLoading existing cleaned data...")
orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")
print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
print(f"Cleaned Performance: {perf_cleaned.count():,} records")


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Clean bronze data
print("\nCleaning data from scratch...")

# Load bronze data if not already loaded
if 'origination_df' not in dir():
    origination_df = spark.read.parquet(paths.origination_bronze)
if 'performance_df' not in dir():
    performance_df = spark.read.parquet(paths.performance_bronze)

# Clean data
orig_cleaned, perf_cleaned = cleaner.clean_both_datasets(
    origination_df, performance_df
)

# Save to silver layer
orig_cleaned.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.silver_dir}/origination_cleaned.parquet")

perf_cleaned.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
print(f"Cleaned Performance: {perf_cleaned.count():,} records")
"""



MODULE 2: DATA CLEANING

Loading existing cleaned data...
Cleaned Origination: 700,000 loans
Cleaned Performance: 43,612,276 records


'\n# PROCESS FROM SCRATCH - Clean bronze data\nprint("\nCleaning data from scratch...")\n\n# Load bronze data if not already loaded\nif \'origination_df\' not in dir():\n    origination_df = spark.read.parquet(paths.origination_bronze)\nif \'performance_df\' not in dir():\n    performance_df = spark.read.parquet(paths.performance_bronze)\n\n# Clean data\norig_cleaned, perf_cleaned = cleaner.clean_both_datasets(\n    origination_df, performance_df\n)\n\n# Save to silver layer\norig_cleaned.write.mode("overwrite")     .option("compression", "snappy")     .parquet(f"{paths.silver_dir}/origination_cleaned.parquet")\n\nperf_cleaned.write.mode("overwrite")     .option("compression", "snappy")     .parquet(f"{paths.silver_dir}/performance_cleaned.parquet")\n\nprint(f"Cleaned Origination: {orig_cleaned.count():,} loans")\nprint(f"Cleaned Performance: {perf_cleaned.count():,} records")\n'

In [8]:

# =============================================================================
# MODULE 3: FEATURE ENGINEERING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 3: FEATURE ENGINEERING")
print("=" * 80)

from features.behavioral_features import BehavioralFeatureEngineer

# Initialize feature engineer
feature_engineer = BehavioralFeatureEngineer(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING FEATURES (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING FEATURES
print("\nLoading existing feature data...")
feature_df = spark.read.parquet(paths.feature_dataset)
print(f"Features loaded: {feature_df.count():,} records")
print(f"Feature count: {len(feature_df.columns)}")


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Create features
print("\nCreating features from scratch...")

# Load cleaned data if not already loaded
if 'orig_cleaned' not in dir():
    orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
if 'perf_cleaned' not in dir():
    perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

# Create features
feature_df = feature_engineer.create_all_features(
    orig_cleaned, perf_cleaned
)

# Remove duplicate columns
seen = set()
cols_to_keep = []
for col_name in feature_df.columns:
    if col_name not in seen:
        seen.add(col_name)
        cols_to_keep.append(col_name)
feature_df = feature_df.select(*cols_to_keep)

# Save feature dataset
feature_df.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(paths.feature_dataset)

print(f"Features created: {feature_df.count():,} records")
print(f"Feature count: {len(feature_df.columns)}")
"""



MODULE 3: FEATURE ENGINEERING

Loading existing feature data...
Features loaded: 43,612,276 records
Feature count: 114


'\n# PROCESS FROM SCRATCH - Create features\nprint("\nCreating features from scratch...")\n\n# Load cleaned data if not already loaded\nif \'orig_cleaned\' not in dir():\n    orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")\nif \'perf_cleaned\' not in dir():\n    perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")\n\n# Create features\nfeature_df = feature_engineer.create_all_features(\n    orig_cleaned, perf_cleaned\n)\n\n# Remove duplicate columns\nseen = set()\ncols_to_keep = []\nfor col_name in feature_df.columns:\n    if col_name not in seen:\n        seen.add(col_name)\n        cols_to_keep.append(col_name)\nfeature_df = feature_df.select(*cols_to_keep)\n\n# Save feature dataset\nfeature_df.write.mode("overwrite")     .option("compression", "snappy")     .parquet(paths.feature_dataset)\n\nprint(f"Features created: {feature_df.count():,} records")\nprint(f"Feature count: {len(feature_df.columns)}")\n'

In [9]:

# =============================================================================
# MODULE 4: TARGET CREATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 4: TARGET CREATION")
print("=" * 80)

from target.target_creation import TargetCreator

# Initialize target creator
target_creator = TargetCreator(
    spark=spark,
    default_threshold=model_config.default_threshold,
    lookahead_months=model_config.lookahead_months
)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING DATASET WITH TARGET (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING DATASET WITH TARGET
print("\nLoading existing dataset with target...")
dataset_df = spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")
print(f"Dataset loaded: {dataset_df.count():,} records")
target_creator.analyze_target_distribution(dataset_df)


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Create target
print("\nCreating target from scratch...")

# Load feature data if not already loaded
if 'feature_df' not in dir():
    feature_df = spark.read.parquet(paths.feature_dataset)

# Create target
dataset_df = target_creator.create_target(feature_df)

# Analyze target distribution
target_creator.analyze_target_distribution(dataset_df)

# Save dataset
dataset_df.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.features_dir}/dataset_with_target.parquet")

print(f"Dataset with target saved: {dataset_df.count():,} records")
"""



MODULE 4: TARGET CREATION

Loading existing dataset with target...
Dataset loaded: 7,223,880 records


'\n# PROCESS FROM SCRATCH - Create target\nprint("\nCreating target from scratch...")\n\n# Load feature data if not already loaded\nif \'feature_df\' not in dir():\n    feature_df = spark.read.parquet(paths.feature_dataset)\n\n# Create target\ndataset_df = target_creator.create_target(feature_df)\n\n# Analyze target distribution\ntarget_creator.analyze_target_distribution(dataset_df)\n\n# Save dataset\ndataset_df.write.mode("overwrite")     .option("compression", "snappy")     .parquet(f"{paths.features_dir}/dataset_with_target.parquet")\n\nprint(f"Dataset with target saved: {dataset_df.count():,} records")\n'

In [10]:

# =============================================================================
# MODULE 5: DATASET CREATION & SPLIT
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 5: DATASET CREATION & SPLIT")
print("=" * 80)

from validation.splitter import DataSplitter

# Initialize splitter
splitter = DataSplitter(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING SPLITS (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING SPLITS
print("\nLoading existing data splits...")

train_df = spark.read.parquet(paths.train_data)
val_df = spark.read.parquet(paths.val_data)
test_df = spark.read.parquet(paths.test_data)

print(f"Train: {train_df.count():,} records")
print(f"Validation: {val_df.count():,} records")
print(f"Test: {test_df.count():,} records")


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------

# # PROCESS FROM SCRATCH - Split data
# print("\nSplitting data from scratch...")

# # Load dataset with target if not already loaded
# if 'dataset_df' not in dir():
#     dataset_df = spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")

# # Split data
# train_df, val_df, test_df = splitter.split_data(
#     dataset_df,
#     train_start_year=model_config.train_start_year,
#     train_end_year=model_config.train_end_year,
#     test_start_year=model_config.test_start_year,
#     test_end_year=model_config.test_end_year,
#     val_frac=model_config.val_frac
# )

# # Save splits
# train_df.write.mode("overwrite").parquet(paths.train_data)
# val_df.write.mode("overwrite").parquet(paths.val_data)
# test_df.write.mode("overwrite").parquet(paths.test_data)

# print(f"Train: {train_df.count():,} records")
# print(f"Validation: {val_df.count():,} records")
# print(f"Test: {test_df.count():,} records")




MODULE 5: DATASET CREATION & SPLIT

Loading existing data splits...
Train: 3,528,464 records
Validation: 887,771 records
Test: 2,241,019 records


In [11]:

# =============================================================================
# MODULE 6: PREPARE PANDAS DATA FOR MODELING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 6: PREPARE PANDAS DATA")
print("=" * 80)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING PANDAS DATA (Skip Processing)
# -----------------------------------------------------------------------------

# LOAD EXISTING PANDAS DATA
print("\nLoading existing pandas data...")

pkl_dir = f"{paths.features_dir}/pandas"
X_train = pd.read_pickle(f"{pkl_dir}/X_train.pkl")
y_train = pd.read_pickle(f"{pkl_dir}/y_train.pkl")
X_val = pd.read_pickle(f"{pkl_dir}/X_val.pkl")
y_val = pd.read_pickle(f"{pkl_dir}/y_val.pkl")
X_test = pd.read_pickle(f"{pkl_dir}/X_test.pkl")
y_test = pd.read_pickle(f"{pkl_dir}/y_test.pkl")

print(f"Train: {len(X_train):,} records")
print(f"Validation: {len(X_val):,} records")
print(f"Test: {len(X_test):,} records")
print(f"Features: {X_train.shape[1]}")


# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------

# # PROCESS FROM SCRATCH - Prepare pandas data
# print("\nPreparing pandas data from scratch...")

# # Load splits if not already loaded
# if 'train_df' not in dir():
#     train_df = spark.read.parquet(paths.train_data)
#     val_df = spark.read.parquet(paths.val_data)
#     test_df = spark.read.parquet(paths.test_data)

# def prepare_features(df):
#     """Prepare features from Spark DataFrame."""
#     feature_cols = [c for c in df.columns if c not in 
#                    ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'target']]
#     pdf = df.select(*feature_cols).toPandas()
#     pdf = pdf.replace([np.inf, -np.inf], np.nan)
#     pdf.fillna(0, inplace=True)
#     return pdf

# X_train = prepare_features(train_df)
# y_train = train_df.select("target").toPandas()["target"]
# X_val = prepare_features(val_df)
# y_val = val_df.select("target").toPandas()["target"]
# X_test = prepare_features(test_df)
# y_test = test_df.select("target").toPandas()["target"]

# # Save pandas data
# pkl_dir = f"{paths.features_dir}/pandas"
# os.makedirs(pkl_dir, exist_ok=True)
# pd.to_pickle(X_train, f"{pkl_dir}/X_train.pkl")
# pd.to_pickle(y_train, f"{pkl_dir}/y_train.pkl")
# pd.to_pickle(X_val, f"{pkl_dir}/X_val.pkl")
# pd.to_pickle(y_val, f"{pkl_dir}/y_val.pkl")
# pd.to_pickle(X_test, f"{pkl_dir}/X_test.pkl")
# pd.to_pickle(y_test, f"{pkl_dir}/y_test.pkl")

# print(f"Train: {len(X_train):,} records")
# print(f"Validation: {len(X_val):,} records")
# print(f"Test: {len(X_test):,} records")
# print(f"Features: {X_train.shape[1]}")



MODULE 6: PREPARE PANDAS DATA

Loading existing pandas data...
Train: 3,528,464 records
Validation: 887,771 records
Test: 2,241,019 records
Features: 119


In [12]:
# =============================================================================
# MODULE 7: HYPERPARAMETER TUNING (Complete with Imports)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 7: HYPERPARAMETER TUNING")
print("=" * 80)

# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

# Import configuration
from config.config import config
paths = config['paths']
model_config = config['model']
feature_config = config['features']

# Import model classes
from models.logistic import LogisticRegressionModel
from models.random_forest import RandomForestModel
from models.xgboost_model import XGBoostModel
from models.lightgbm_model import LightGBMModel
from models.catboost_model import CatBoostModel
from models.ensemble import StackingEnsemble
from models.hyperparameter_tuning import HyperparameterTuner

# Check and install optuna if needed
try:
    import optuna
    print("✅ Optuna is available")
except ImportError:
    print("❌ Optuna is NOT available. Installing...")
    !pip install optuna
    import optuna

# Check for other required packages
try:
    import xgboost
    print("✅ XGBoost is available")
except ImportError:
    print("⚠️ XGBoost not available")

try:
    import lightgbm
    print("✅ LightGBM is available")
except ImportError:
    print("⚠️ LightGBM not available")

try:
    import catboost
    print("✅ CatBoost is available")
except ImportError:
    print("⚠️ CatBoost not available")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# =============================================================================
# STEP 1: Use Existing Spark Session
# =============================================================================

print("\nStep 1: Using existing Spark session...")

# Check if Spark session exists
try:
    spark
    print("✅ Existing Spark session found")
except NameError:
    print("Creating new Spark session...")
    from data_ingestion.data_ingestion import create_spark_session
    spark = create_spark_session()

# =============================================================================
# STEP 2: Load Data Using SQL
# =============================================================================

print("\nStep 2: Loading and sampling data...")

# Register DataFrames as temporary views
try:
    train_df.createOrReplaceTempView("train_data")
    val_df.createOrReplaceTempView("val_data")
    test_df.createOrReplaceTempView("test_data")
    print("✅ DataFrames registered as views")
except Exception as e:
    print(f"⚠️ Could not register views: {e}")
    print("Attempting to load from parquet...")
    try:
        train_df = spark.read.parquet(paths.train_data)
        val_df = spark.read.parquet(paths.val_data)
        test_df = spark.read.parquet(paths.test_data)
        train_df.createOrReplaceTempView("train_data")
        val_df.createOrReplaceTempView("val_data")
        test_df.createOrReplaceTempView("test_data")
        print("✅ DataFrames loaded from parquet and registered")
    except Exception as e2:
        print(f"❌ Could not load data: {e2}")
        raise

# Sample data (20% of training data for tuning)
sample_fraction = 0.20
print(f"Sampling {sample_fraction*100:.0f}% of training data...")

# Sample training data
train_sample_sql = f"""
SELECT * FROM train_data 
WHERE RAND() < {sample_fraction}
"""
train_sample_df = spark.sql(train_sample_sql)
sample_count = train_sample_df.count()
print(f"Sampled {sample_count:,} training records")

# Sample validation data
val_sample_sql = f"""
SELECT * FROM val_data 
WHERE RAND() < {sample_fraction}
"""
val_sample_df = spark.sql(val_sample_sql)
if val_sample_df.count() > 0:
    print(f"Sampled {val_sample_df.count():,} validation records")
else:
    val_sample_df = None

# =============================================================================
# STEP 3: Convert to Pandas
# =============================================================================

print("\nStep 3: Converting to Pandas...")

# Disable Arrow for stability
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

def safe_to_pandas(df):
    """Safely convert Spark DataFrame to Pandas."""
    try:
        return df.toPandas()
    except Exception as e:
        print(f"Conversion failed: {e}")
        return df.toPandas()

# Drop target column from features
drop_cols = ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'target']

# Convert to pandas
X_train_pd = safe_to_pandas(train_sample_df.drop(*drop_cols))
y_train_pd = safe_to_pandas(train_sample_df.select('target'))['target']

if val_sample_df is not None:
    X_val_pd = safe_to_pandas(val_sample_df.drop(*drop_cols))
    y_val_pd = safe_to_pandas(val_sample_df.select('target'))['target']
else:
    X_val_pd = None
    y_val_pd = None

print(f"Train features shape: {X_train_pd.shape}")
print(f"Train target shape: {y_train_pd.shape}")
if X_val_pd is not None:
    print(f"Validation features shape: {X_val_pd.shape}")

# Re-enable Arrow
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# =============================================================================
# STEP 4: Simple Data Preprocessing
# =============================================================================

print("\nStep 4: Preprocessing data...")

def simple_preprocess(df):
    """Simple preprocessing that handles all data types."""
    from sklearn.preprocessing import LabelEncoder
    df_processed = df.copy()
    
    for col in df_processed.columns:
        if df_processed[col].dtype == 'object' or df_processed[col].dtype == 'category':
            df_processed[col] = df_processed[col].astype(str)
            df_processed[col] = df_processed[col].replace('nan', 'MISSING')
            df_processed[col] = df_processed[col].replace('None', 'MISSING')
            
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col])
        else:
            df_processed[col] = df_processed[col].fillna(0)
    
    df_processed = df_processed.apply(pd.to_numeric, errors='coerce')
    df_processed = df_processed.fillna(0)
    
    return df_processed

# Preprocess data
X_train_processed = simple_preprocess(X_train_pd)
if X_val_pd is not None:
    X_val_processed = simple_preprocess(X_val_pd)
else:
    X_val_processed = None

print(f"Processed training shape: {X_train_processed.shape}")

# =============================================================================
# STEP 5: User Configuration - TUNING MODE SELECTION
# =============================================================================

print("\n" + "=" * 80)
print("TUNING MODE SELECTION")
print("=" * 80)

# ================================================================
# CONFIGURATION OPTIONS - UNCOMMENT THE ONE YOU WANT
# ================================================================

# OPTION 1: LOAD EXISTING TUNED PARAMETERS (Skip all tuning)
# -------------------------------------------------------------
# TUNING_MODE = "LOAD_EXISTING"  # Uncomment this to load existing

# OPTION 2: RETUNE ALL MODELS (Run tuning for all models)
# -------------------------------------------------------------
# TUNING_MODE = "TUNE_ALL"  # Uncomment this to retune all models

# OPTION 3: RETUNE SELECTED MODELS (Specify which models to tune)
# -------------------------------------------------------------
# TUNING_MODE = "TUNE_SELECTED"
# MODELS_TO_TUNE = ['catboost']  # Options: 'logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost'
# MODELS_TO_TUNE = ['logistic', 'xgboost']
# MODELS_TO_TUNE = ['catboost', 'lightgbm']

# ================================================================
# CURRENT CONFIGURATION - Set this to your desired mode
# ================================================================

# Uncomment ONE of the following lines:
# TUNING_MODE = "LOAD_EXISTING"    # OPTION 1: Load existing (skip tuning)
# TUNING_MODE = "TUNE_ALL"        # OPTION 2: Retune all models
TUNING_MODE = "TUNE_SELECTED"   # OPTION 3: Retune selected models

# If using TUNE_SELECTED, specify which models:
MODELS_TO_TUNE = ['catboost']    # Add models you want to tune

print(f"\nSelected Tuning Mode: {TUNING_MODE}")
if TUNING_MODE == "TUNE_SELECTED":
    print(f"  Models to tune: {MODELS_TO_TUNE}")
print("-" * 80)

# =============================================================================
# STEP 6: Load Existing Tuned Parameters (If exists)
# =============================================================================

print("\nStep 5: Loading existing tuned parameters...")

tuned_params_path = f"{paths.results_dir}/tuned_params.json"
existing_params = {}

if os.path.exists(tuned_params_path):
    with open(tuned_params_path, 'r') as f:
        existing_params = json.load(f)
    print(f"✅ Loaded existing tuned parameters with models: {list(existing_params.keys())}")
else:
    print("⚠️ No existing tuned parameters found. Will create new.")

# =============================================================================
# STEP 7: Hyperparameter Tuning Based on Mode
# =============================================================================

print("\n" + "=" * 80)
print(f"TUNING MODE: {TUNING_MODE}")
print("=" * 80)

# Create tuner
tuner = HyperparameterTuner(
    n_trials=10,  # Adjust as needed
    cv_folds=3,
    random_state=model_config.random_state
)

tuned_params = existing_params.copy()  # Start with existing params

# =============================================================================
# MODE 1: LOAD EXISTING (Skip all tuning)
# =============================================================================

if TUNING_MODE == "LOAD_EXISTING":
    print("\n📂 MODE: LOAD EXISTING PARAMETERS")
    print("Skipping all tuning. Using existing parameters.")
    
    if tuned_params:
        print("\nLoaded parameters:")
        for model, params in tuned_params.items():
            if params:
                print(f"  ✅ {model}: {list(params.keys())}")
            else:
                print(f"  ⚠️ {model}: No parameters")
    else:
        print("\n⚠️ No existing parameters found. You may want to run tuning.")

# =============================================================================
# MODE 2: TUNE ALL MODELS
# =============================================================================

elif TUNING_MODE == "TUNE_ALL":
    print("\n🔄 MODE: TUNE ALL MODELS")
    print("Running hyperparameter tuning for all models...")
    
    # 1. Logistic Regression
    print("\nTuning Logistic Regression...")
    try:
        params = tuner.tune_logistic_regression(X_train_processed, y_train_pd)
        tuned_params['logistic'] = params
        print(f"  ✅ Logistic Regression tuning completed")
    except Exception as e:
        print(f"  ⚠️ Could not tune logistic: {e}")
        tuned_params['logistic'] = {'C': 1.0, 'solver': 'lbfgs', 'class_weight': 'balanced'}

    # 2. Random Forest
    print("\nTuning Random Forest...")
    try:
        params = tuner.tune_random_forest(X_train_processed, y_train_pd)
        tuned_params['random_forest'] = params
        print(f"  ✅ Random Forest tuning completed")
    except Exception as e:
        print(f"  ⚠️ Could not tune random_forest: {e}")
        tuned_params['random_forest'] = {}

    # 3. XGBoost
    try:
        import xgboost
        print("\nTuning XGBoost...")
        params = tuner.tune_xgboost(X_train_processed, y_train_pd)
        tuned_params['xgboost'] = params
        print(f"  ✅ XGBoost tuning completed")
    except ImportError:
        print("⚠️ XGBoost not available, skipping...")
        tuned_params['xgboost'] = {}
    except Exception as e:
        print(f"  ⚠️ Could not tune xgboost: {e}")
        tuned_params['xgboost'] = {}

    # 4. LightGBM
    try:
        import lightgbm
        print("\nTuning LightGBM...")
        params = tuner.tune_lightgbm(X_train_processed, y_train_pd)
        tuned_params['lightgbm'] = params
        print(f"  ✅ LightGBM tuning completed")
    except ImportError:
        print("⚠️ LightGBM not available, skipping...")
        tuned_params['lightgbm'] = {}
    except Exception as e:
        print(f"  ⚠️ Could not tune lightgbm: {e}")
        tuned_params['lightgbm'] = {}

    # 5. CatBoost
    try:
        import catboost
        print("\nTuning CatBoost...")
        # Use original data for CatBoost (with categorical features)
        cat_cols = X_train_pd.select_dtypes(include=['object', 'category']).columns.tolist()
        params = tuner.tune_catboost(X_train_pd, y_train_pd)
        tuned_params['catboost'] = params
        print(f"  ✅ CatBoost tuning completed")
    except ImportError:
        print("⚠️ CatBoost not available, skipping...")
        tuned_params['catboost'] = {}
    except Exception as e:
        print(f"  ⚠️ Could not tune catboost: {e}")
        tuned_params['catboost'] = {
            'iterations': 300,
            'depth': 6,
            'learning_rate': 0.05,
            'l2_leaf_reg': 3,
            'border_count': 254,
            'auto_class_weights': 'Balanced'
        }

# =============================================================================
# MODE 3: TUNE SELECTED MODELS ONLY
# =============================================================================

elif TUNING_MODE == "TUNE_SELECTED":
    print(f"\n🎯 MODE: TUNE SELECTED MODELS")
    print(f"  Models to tune: {MODELS_TO_TUNE}")
    
    # Check which models to tune
    model_tuning_functions = {
        'logistic': {
            'func': tuner.tune_logistic_regression,
            'data': X_train_processed,
            'fallback': {'C': 1.0, 'solver': 'lbfgs', 'class_weight': 'balanced'}
        },
        'random_forest': {
            'func': tuner.tune_random_forest,
            'data': X_train_processed,
            'fallback': {}
        },
        'xgboost': {
            'func': tuner.tune_xgboost,
            'data': X_train_processed,
            'fallback': {}
        },
        'lightgbm': {
            'func': tuner.tune_lightgbm,
            'data': X_train_processed,
            'fallback': {}
        },
        'catboost': {
            'func': tuner.tune_catboost,
            'data': X_train_pd,  # CatBoost needs original data with categoricals
            'fallback': {
                'iterations': 300,
                'depth': 6,
                'learning_rate': 0.05,
                'l2_leaf_reg': 3,
                'border_count': 254,
                'auto_class_weights': 'Balanced'
            }
        }
    }
    
    for model_name in MODELS_TO_TUNE:
        if model_name in model_tuning_functions:
            print(f"\nTuning {model_name}...")
            
            # Check if model package is available
            if model_name == 'xgboost':
                try:
                    import xgboost
                except ImportError:
                    print(f"  ⚠️ XGBoost not available, skipping...")
                    continue
            elif model_name == 'lightgbm':
                try:
                    import lightgbm
                except ImportError:
                    print(f"  ⚠️ LightGBM not available, skipping...")
                    continue
            elif model_name == 'catboost':
                try:
                    import catboost
                except ImportError:
                    print(f"  ⚠️ CatBoost not available, skipping...")
                    continue
            
            try:
                info = model_tuning_functions[model_name]
                params = info['func'](info['data'], y_train_pd)
                tuned_params[model_name] = params
                print(f"  ✅ {model_name} tuning completed")
            except Exception as e:
                print(f"  ⚠️ Could not tune {model_name}: {e}")
                print(f"  Using default parameters for {model_name}")
                tuned_params[model_name] = model_tuning_functions[model_name]['fallback']
        else:
            print(f"  ⚠️ Unknown model: {model_name}")

# =============================================================================
# STEP 8: Save Tuned Parameters
# =============================================================================

print("\nStep 6: Saving tuned parameters...")

# Save updated params
os.makedirs(paths.results_dir, exist_ok=True)
with open(tuned_params_path, 'w') as f:
    json.dump(tuned_params, f, indent=2)

print(f"✅ Parameters saved to: {tuned_params_path}")

# =============================================================================
# STEP 9: Display Results
# =============================================================================

print("\n" + "=" * 80)
print("FINAL TUNED PARAMETERS SUMMARY")
print("=" * 80)

if tuned_params:
    print("\nTuned parameters by model:")
    print("-" * 60)
    for model, params in tuned_params.items():
        if params:
            print(f"\n  {model}:")
            for key, value in params.items():
                print(f"    {key}: {value}")
        else:
            print(f"\n  {model}: No parameters tuned")
else:
    print("\n⚠️ No parameters found. Tuning may have failed.")

# =============================================================================
# STEP 10: Clean up
# =============================================================================

print("\nStep 7: Cleaning up memory...")

# Delete large DataFrames
try:
    del X_train_pd, y_train_pd, X_train_processed
    if X_val_pd is not None:
        del X_val_pd, y_val_pd
    if X_val_processed is not None:
        del X_val_processed
except:
    pass

import gc
gc.collect()

print("Memory cleanup completed!")

# =============================================================================
# STEP 11: Keep Spark session active
# =============================================================================

print("\n✅ Spark session is still active for subsequent modules.")
print("To stop Spark, run: spark.stop()")


MODULE 7: HYPERPARAMETER TUNING
✅ Optuna is available
✅ XGBoost is available
✅ LightGBM is available
✅ CatBoost is available

Step 1: Using existing Spark session...
✅ Existing Spark session found

Step 2: Loading and sampling data...
✅ DataFrames registered as views
Sampling 20% of training data...
Sampled 706,230 training records
Sampled 177,489 validation records

Step 3: Converting to Pandas...
Train features shape: (706230, 119)
Train target shape: (706230,)
Validation features shape: (177489, 119)

Step 4: Preprocessing data...
Processed training shape: (706230, 119)

TUNING MODE SELECTION

Selected Tuning Mode: TUNE_SELECTED
  Models to tune: ['catboost']
--------------------------------------------------------------------------------

Step 5: Loading existing tuned parameters...
✅ Loaded existing tuned parameters with models: ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost']

TUNING MODE: TUNE_SELECTED

🎯 MODE: TUNE SELECTED MODELS
  Models to tune: ['catboost']

[I 2026-07-10 01:37:13,670] A new study created in memory with name: catboost
[I 2026-07-10 01:39:41,771] Trial 0 finished with value: 1.0 and parameters: {'iterations': 100, 'depth': 7, 'learning_rate': 0.011784979387549556, 'l2_leaf_reg': 4.399422004635733, 'border_count': 114, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 1.0.
[I 2026-07-10 01:45:49,228] Trial 1 finished with value: 1.0 and parameters: {'iterations': 350, 'depth': 4, 'learning_rate': 0.029725561877760476, 'l2_leaf_reg': 8.765165379340747, 'border_count': 148, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 1.0.
[I 2026-07-10 01:48:50,197] Trial 2 finished with value: 1.0 and parameters: {'iterations': 100, 'depth': 9, 'learning_rate': 0.04945936718091979, 'l2_leaf_reg': 3.0836234093620396, 'border_count': 158, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 1.0.
[I 2026-07-10 01:51:47,956] Trial 3 finished with value: 1.0 and parameters: {'iterations': 100, 'depth':

  ✅ catboost tuning completed

Step 6: Saving tuned parameters...
✅ Parameters saved to: D:/Projects/credit_risk_scoring/results/tuned_params.json

FINAL TUNED PARAMETERS SUMMARY

Tuned parameters by model:
------------------------------------------------------------

  logistic:
    C: 0.0024200226408001837
    solver: newton-cg
    class_weight: None

  random_forest:
    n_estimators: 250
    max_depth: 4
    min_samples_split: 120
    min_samples_leaf: 45
    max_features: None
    class_weight: None

  xgboost:
    n_estimators: 400
    max_depth: 8
    learning_rate: 0.11443003152089135
    subsample: 0.6671834049285933
    colsample_bytree: 0.8259247242146994
    scale_pos_weight: 8.149018551350736
    tree_method: approx

  lightgbm:
    n_estimators: 250
    num_leaves: 99
    max_depth: 9
    learning_rate: 0.21214060602402027
    feature_fraction: 0.5659896413977688
    bagging_fraction: 0.9249802704525223
    bagging_freq: 2
    is_unbalance: True

  catboost:
    iteration

In [16]:
import models.hyperparameter_tuning as ht
print(ht.__file__)

d:\Projects\credit_risk_scoring\models\hyperparameter_tuning.py


In [4]:
# =============================================================================
# MODULE 8: MODEL TRAINING (PySpark ML - Fixed NaN Handling)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 8: MODEL TRAINING (PySpark ML)")
print("=" * 80)

import os
import json
import pickle
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand, lit, isnan, isnull, count
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler, Imputer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# =============================================================================
# STEP 1: Use Existing Spark Session
# =============================================================================

print("\nStep 1: Using existing Spark session...")

try:
    spark
    print("✅ Existing Spark session found")
except NameError:
    print("Creating new Spark session...")
    from data_ingestion.data_ingestion import create_spark_session
    spark = create_spark_session()

# =============================================================================
# STEP 2: Load Data
# =============================================================================

print("\nStep 2: Loading data...")

# Load data if not already loaded
try:
    train_df
    print("✅ Data already loaded")
except NameError:
    print("Loading data from parquet...")
    train_df = spark.read.parquet(paths.train_data)
    val_df = spark.read.parquet(paths.val_data)
    test_df = spark.read.parquet(paths.test_data)

print(f"Training records: {train_df.count():,}")
print(f"Validation records: {val_df.count():,}")
print(f"Test records: {test_df.count():,}")

# =============================================================================
# STEP 3: Remove Data Leakage Columns
# =============================================================================

print("\nStep 3: Removing data leakage columns...")

leakage_cols = [
    'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'ZERO_BALANCE_REMOVAL_UPB',
    'ACTUAL_LOSS_CALCULATION', 'DELINQUENT_ACCRUED_INTEREST',
    'MI_RECOVERIES', 'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES',
    'TOTAL_EXPENSES', 'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS',
    'TAXES_AND_INSURANCE', 'MISCELLANEOUS_EXPENSES',
    'CURRENT_MONTH_MODIFICATION_COST', 'CUMULATIVE_MODIFICATION_COST',
    'DDLPI', 'DEFECT_SETTLEMENT_DATE', 'PRE_RELIEF_REFINANCE_LSN',
    'SPECIAL_ELIGIBILITY_PROGRAM', 'PROPERTY_VALUATION_METHOD',
    'MI_CANCELLATION_INDICATOR', 'MSA'
]

# Drop leakage columns
drop_cols = [c for c in leakage_cols if c in train_df.columns]
if drop_cols:
    print(f"  Dropping {len(drop_cols)} leakage columns")
    train_df = train_df.drop(*drop_cols)
    val_df = val_df.drop(*drop_cols)
    test_df = test_df.drop(*drop_cols)

# Also drop non-feature columns
non_feature_cols = ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD']
train_df = train_df.drop(*non_feature_cols)
val_df = val_df.drop(*non_feature_cols)
test_df = test_df.drop(*non_feature_cols)

print(f"Training data now has {len(train_df.columns)} columns")

# =============================================================================
# STEP 4: Sample Data (Optional - for faster training)
# =============================================================================

print("\nStep 4: Sampling data (optional)...")

# Use 30% of data for faster training (adjust based on your needs)
use_sample = True
sample_fraction = 0.30  # 30% of data

if use_sample:
    print(f"Using {sample_fraction*100:.0f}% of data...")
    train_df = train_df.sample(fraction=sample_fraction, seed=42)
    val_df = val_df.sample(fraction=sample_fraction, seed=42)
    test_df = test_df.sample(fraction=sample_fraction, seed=42)
    
    # Cache sampled data
    train_df.cache()
    val_df.cache()
    test_df.cache()
    
    print(f"Training records: {train_df.count():,}")
    print(f"Validation records: {val_df.count():,}")
    print(f"Test records: {test_df.count():,}")

# =============================================================================
# STEP 5: Check and Handle NaN Values
# =============================================================================

print("\nStep 5: Checking and handling NaN values...")

# Function to count NaNs in a DataFrame
def count_nulls(df):
    return df.select([count(when(isnan(c) | isnull(c), c)).alias(c) for c in df.columns])

# Check for NaNs
print("  Checking for NaN values in training data...")
null_counts = count_nulls(train_df).collect()[0]
null_cols = []
for col_name, null_count in null_counts.asDict().items():
    if null_count > 0:
        null_cols.append(col_name)
        print(f"    {col_name}: {null_count:,} NaNs")

if null_cols:
    print(f"\n  Found {len(null_cols)} columns with NaN values")
    
    # Separate numeric and categorical columns
    numeric_cols = []
    categorical_cols = []
    
    # Get column types
    for col_name in train_df.columns:
        if col_name != 'target':
            col_type = dict(train_df.dtypes)[col_name]
            if col_type in ['double', 'float', 'int', 'bigint']:
                numeric_cols.append(col_name)
            else:
                categorical_cols.append(col_name)
    
    print(f"  Numeric columns: {len(numeric_cols)}")
    print(f"  Categorical columns: {len(categorical_cols)}")
    
    # Step 6: Impute NaN values
    print("\nStep 6: Imputing NaN values...")
    
    # For numeric columns: impute with median
    imputers = []
    for col_name in numeric_cols:
        # Check if column has nulls
        null_count = train_df.filter(isnan(col_name) | isnull(col_name)).count()
        if null_count > 0:
            imputer = Imputer(
                inputCols=[col_name],
                outputCols=[col_name],
                strategy="median"
            )
            imputers.append(imputer)
    
    if imputers:
        print(f"  Imputing {len(imputers)} numeric columns with median...")
        imputer_pipeline = Pipeline(stages=imputers)
        imputer_model = imputer_pipeline.fit(train_df)
        train_df = imputer_model.transform(train_df)
        val_df = imputer_model.transform(val_df)
        test_df = imputer_model.transform(test_df)
    
    # For categorical columns: fill with 'MISSING'
    for col_name in categorical_cols:
        train_df = train_df.fillna('MISSING', subset=[col_name])
        val_df = val_df.fillna('MISSING', subset=[col_name])
        test_df = test_df.fillna('MISSING', subset=[col_name])
    
    print("  ✅ NaN values handled")
else:
    print("  ✅ No NaN values found")

# =============================================================================
# STEP 7: Feature Engineering with PySpark
# =============================================================================

print("\nStep 7: Feature engineering...")

# Identify categorical and numeric columns
all_cols = train_df.columns
categorical_cols = []
numeric_cols = []

# Sample to check column types
sample = train_df.limit(100).toPandas()
for col in all_cols:
    if col != 'target':
        if sample[col].dtype == 'object':
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

print(f"  Categorical columns: {len(categorical_cols)}")
print(f"  Numeric columns: {len(numeric_cols)}")

# =============================================================================
# STEP 8: Build PySpark ML Pipeline
# =============================================================================

print("\nStep 8: Building PySpark ML pipeline...")

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# Stage 1: Handle categorical columns with StringIndexer
indexers = []
for col in categorical_cols:
    indexer = StringIndexer(
        inputCol=col, 
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    indexers.append(indexer)

# Stage 2: Assemble all features
feature_cols = [f"{col}_indexed" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"  # This handles remaining NaN values
)

# Stage 3: Scale features (optional)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False  # Sparse features with mean can be expensive
)

# =============================================================================
# STEP 9: Train Logistic Regression
# =============================================================================

print("\nStep 9: Training Logistic Regression...")

# Create pipeline for Logistic Regression
lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="target",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True
)

lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

try:
    lr_model = lr_pipeline.fit(train_df)
    print("  ✅ Logistic Regression trained")
    
    # Evaluate
    lr_pred = lr_model.transform(val_df)
    evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="prediction")
    lr_auc = evaluator.evaluate(lr_pred)
    print(f"  ✅ Logistic Regression AUC: {lr_auc:.4f}")
    
    # Save model
    lr_model.write().overwrite().save(f"{paths.models_dir}/spark_lr_model")
    print(f"  ✅ Saved Logistic Regression model")
    
except Exception as e:
    print(f"  ❌ Logistic Regression failed: {e}")
    import traceback
    traceback.print_exc()

# =============================================================================
# STEP 10: Train Random Forest
# =============================================================================

print("\nStep 10: Training Random Forest...")

rf = RandomForestClassifier(
    featuresCol="features",  # Random Forest doesn't need scaling
    labelCol="target",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=50,
    seed=42,
    subsamplingRate=0.5,
    impurity="gini"
)

rf_pipeline = Pipeline(stages=indexers + [assembler, rf])

try:
    rf_model = rf_pipeline.fit(train_df)
    print("  ✅ Random Forest trained")
    
    rf_pred = rf_model.transform(val_df)
    rf_auc = evaluator.evaluate(rf_pred)
    print(f"  ✅ Random Forest AUC: {rf_auc:.4f}")
    
    # Save model
    rf_model.write().overwrite().save(f"{paths.models_dir}/spark_rf_model")
    print(f"  ✅ Saved Random Forest model")
    
except Exception as e:
    print(f"  ❌ Random Forest failed: {e}")
    import traceback
    traceback.print_exc()

# =============================================================================
# STEP 11: Train Gradient Boosted Trees
# =============================================================================

print("\nStep 11: Training Gradient Boosted Trees...")

gbt = GBTClassifier(
    featuresCol="scaled_features",
    labelCol="target",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
    subsamplingRate=0.5
)

gbt_pipeline = Pipeline(stages=indexers + [assembler, scaler, gbt])

try:
    gbt_model = gbt_pipeline.fit(train_df)
    print("  ✅ Gradient Boosted Trees trained")
    
    gbt_pred = gbt_model.transform(val_df)
    gbt_auc = evaluator.evaluate(gbt_pred)
    print(f"  ✅ Gradient Boosted Trees AUC: {gbt_auc:.4f}")
    
    # Save model
    gbt_model.write().overwrite().save(f"{paths.models_dir}/spark_gbt_model")
    print(f"  ✅ Saved Gradient Boosted Trees model")
    
except Exception as e:
    print(f"  ❌ Gradient Boosted Trees failed: {e}")
    import traceback
    traceback.print_exc()

# =============================================================================
# STEP 12: Summary
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING COMPLETED!")
print("=" * 80)

# Summary table
print("\nModel Performance Summary:")
print("-" * 60)
print(f"{'Model':<30} {'AUC':<10}")
print("-" * 60)

if 'lr_auc' in locals():
    print(f"{'Logistic Regression':<30} {lr_auc:.4f}")
if 'rf_auc' in locals():
    print(f"{'Random Forest':<30} {rf_auc:.4f}")
if 'gbt_auc' in locals():
    print(f"{'Gradient Boosted Trees':<30} {gbt_auc:.4f}")
print("-" * 60)

# Clean up
print("\nCleaning up...")
train_df.unpersist()
val_df.unpersist()
test_df.unpersist()

print(f"\nModels saved to: {paths.models_dir}")


MODULE 8: MODEL TRAINING (PySpark ML)

Step 1: Using existing Spark session...
✅ Existing Spark session found

Step 2: Loading data...
Loading data from parquet...
Training records: 3,528,464
Validation records: 887,771
Test records: 2,241,019

Step 3: Removing data leakage columns...
  Dropping 22 leakage columns
Training data now has 98 columns

Step 4: Sampling data (optional)...
Using 30% of data...
Training records: 1,058,239
Validation records: 266,194
Test records: 672,195

Step 5: Checking and handling NaN values...
  Checking for NaN values in training data...
    REMAINING_MONTHS_TO_LEGAL_MATURITY: 8 NaNs
    MODIFICATION_FLAG: 1,058,204 NaNs
    INTEREST_RATE_STEP_INDICATOR: 1,058,204 NaNs
    PAYMENT_DEFERRAL_FLAG: 1,058,239 NaNs
    ELTV: 1,057,984 NaNs
    DELINQUENCY_DUE_TO_DISASTER: 1,058,239 NaNs
    BORROWER_ASSISTANCE_STATUS_CODE: 1,058,239 NaNs
    CREDIT_SCORE: 3,170 NaNs
    MI_PERCENTAGE: 22 NaNs
    NUMBER_OF_UNITS: 16 NaNs
    ORIGINAL_CLTV: 70 NaNs
    ORIGIN

Py4JJavaError: An error occurred while calling o991.fit.
: org.apache.spark.SparkException: surrogate cannot be computed. All the values in ORIGINAL_LTV are Null, Nan or missingValue(NaN)
	at org.apache.spark.ml.feature.Imputer.fit(Imputer.scala:199)
	at org.apache.spark.ml.feature.Imputer.fit(Imputer.scala:115)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j

In [5]:
# =============================================================================
# MODULE 8: MODEL TRAINING (Fixed CatBoost)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 8: MODEL TRAINING")
print("=" * 80)

# Import models
from models.logistic import LogisticRegressionModel
from models.random_forest import RandomForestModel
from models.xgboost_model import XGBoostModel
from models.lightgbm_model import LightGBMModel
from models.catboost_model import CatBoostModel
from models.ensemble import StackingEnsemble

import os
import json
import pickle
import pandas as pd
import numpy as np

# =============================================================================
# STEP 1: Ensure Data is Numeric
# =============================================================================

print("\nStep 1: Ensuring data is numeric...")

# Use the function from tuning step
def simple_preprocess(df):
    from sklearn.preprocessing import LabelEncoder
    df_processed = df.copy()
    
    for col in df_processed.columns:
        if df_processed[col].dtype == 'object' or df_processed[col].dtype == 'category':
            df_processed[col] = df_processed[col].astype(str)
            df_processed[col] = df_processed[col].replace('nan', 'MISSING')
            df_processed[col] = df_processed[col].replace('None', 'MISSING')
            
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col])
        else:
            df_processed[col] = df_processed[col].fillna(0)
    
    df_processed = df_processed.apply(pd.to_numeric, errors='coerce')
    df_processed = df_processed.fillna(0)
    
    return df_processed

# Check if we have processed data from tuning step
if 'X_train_processed' in locals() and X_train_processed is not None:
    print("Using pre-processed data from tuning step")
    X_train_final = X_train_processed
    X_val_final = X_val_processed if 'X_val_processed' in locals() else None
else:
    print("Pre-processing data for training...")
    # Load original data if needed
    if 'X_train' not in locals():
        print("Loading data from parquet...")
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        train_df = spark.read.parquet(paths.train_data)
        val_df = spark.read.parquet(paths.val_data)
        
        # Convert to pandas
        X_train = train_df.drop('target').toPandas()
        y_train = train_df.select('target').toPandas()['target']
        X_val = val_df.drop('target').toPandas()
        y_val = val_df.select('target').toPandas()['target']
    
    X_train_final = simple_preprocess(X_train)
    X_val_final = simple_preprocess(X_val)

print(f"Training data shape: {X_train_final.shape}")
print(f"Training data types: {X_train_final.dtypes.value_counts().to_dict()}")

# Also keep original data for CatBoost
X_train_original = X_train if 'X_train' in locals() else None
X_val_original = X_val if 'X_val' in locals() else None

# =============================================================================
# STEP 2: Load Tuned Parameters
# =============================================================================

print("\nStep 2: Loading tuned parameters...")

tuned_params_path = f"{paths.results_dir}/tuned_params.json"
if os.path.exists(tuned_params_path):
    with open(tuned_params_path, 'r') as f:
        tuned_params = json.load(f)
    print("✅ Loaded tuned parameters")
    for model, params in tuned_params.items():
        if params:
            print(f"  {model}: {list(params.keys())}")
else:
    print("⚠️ No tuned parameters found, using defaults")
    tuned_params = {}

# =============================================================================
# STEP 3: Train Models
# =============================================================================

print("\nStep 3: Training models...")

models = {}

# 1. Logistic Regression
print("\nTraining Logistic Regression...")
try:
    lr = LogisticRegressionModel(
        random_state=model_config.random_state,
        **tuned_params.get('logistic', {})
    )
    lr.fit(X_train_final, y_train)
    models['logistic'] = lr
    print("  ✅ Logistic Regression trained")
except Exception as e:
    print(f"  ❌ Logistic Regression failed: {e}")

# 2. Random Forest
print("\nTraining Random Forest...")
try:
    rf = RandomForestModel(
        random_state=model_config.random_state,
        **tuned_params.get('random_forest', {})
    )
    rf.fit(X_train_final, y_train)
    models['random_forest'] = rf
    print("  ✅ Random Forest trained")
except Exception as e:
    print(f"  ❌ Random Forest failed: {e}")

# 3. XGBoost
print("\nTraining XGBoost...")
try:
    import xgboost
    X_train_num = X_train_final.astype(float)
    X_val_num = X_val_final.astype(float) if X_val_final is not None else None
    
    xgb = XGBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get('xgboost', {})
    )
    xgb.fit(X_train_num, y_train, X_val_num, y_val if y_val is not None else None)
    models['xgboost'] = xgb
    print("  ✅ XGBoost trained")
except Exception as e:
    print(f"  ❌ XGBoost failed: {e}")

# 4. LightGBM
print("\nTraining LightGBM...")
try:
    import lightgbm
    lgb = LightGBMModel(
        random_state=model_config.random_state,
        **tuned_params.get('lightgbm', {})
    )
    lgb.fit(X_train_final, y_train, X_val_final, y_val if y_val is not None else None)
    models['lightgbm'] = lgb
    print("  ✅ LightGBM trained")
except Exception as e:
    print(f"  ❌ LightGBM failed: {e}")

# 5. CatBoost - FIXED: Use original data with proper categorical handling
print("\nTraining CatBoost...")
try:
    import catboost
    
    # Get original data for CatBoost
    if X_train_original is None:
        # Load original data if not available
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        train_df = spark.read.parquet(paths.train_data)
        val_df = spark.read.parquet(paths.val_data)
        X_train_original = train_df.drop('target').toPandas()
        y_train = train_df.select('target').toPandas()['target']
        X_val_original = val_df.drop('target').toPandas()
        y_val = val_df.select('target').toPandas()['target']
    
    # Identify categorical columns for CatBoost
    cat_cols = X_train_original.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"  Categorical columns for CatBoost: {len(cat_cols)}")
    
    # Prepare data for CatBoost (convert categorical to strings)
    def prepare_catboost_data(df):
        df_prep = df.copy()
        for col in cat_cols:
            if col in df_prep.columns:
                df_prep[col] = df_prep[col].fillna('MISSING').astype(str)
                df_prep[col] = df_prep[col].replace('nan', 'MISSING')
                df_prep[col] = df_prep[col].replace('None', 'MISSING')
        return df_prep
    
    # Create CatBoost model
    cat = CatBoostModel(
        random_state=model_config.random_state,
        cat_features=cat_cols if cat_cols else None,
        **tuned_params.get('catboost', {})
    )
    
    # Train on original data (with categoricals as strings)
    X_train_cat = prepare_catboost_data(X_train_original)
    X_val_cat = prepare_catboost_data(X_val_original) if X_val_original is not None else None
    
    cat.fit(X_train_cat, y_train, X_val_cat, y_val if y_val is not None else None)
    models['catboost'] = cat
    print("  ✅ CatBoost trained")
except ImportError:
    print("  ⚠️ CatBoost not available, skipping...")
except Exception as e:
    print(f"  ❌ CatBoost failed: {e}")

# 6. Ensemble - FIXED
if len(models) >= 2:
    print("\nTraining Stacking Ensemble...")
    try:
        # Get available models
        base_models = list(models.values())
        
        # Use LogisticRegressionModel as meta model to avoid parameter issues
        from models.logistic import LogisticRegressionModel
        
        ensemble = StackingEnsemble(
            base_models=base_models,
            meta_model=LogisticRegressionModel(
                random_state=model_config.random_state,
                class_weight='balanced'
            ),
            cv_folds=min(3, model_config.cv_folds),
            random_state=model_config.random_state
        )
        ensemble.fit(X_train_final, y_train, X_val_final, y_val if y_val is not None else None)
        models['ensemble'] = ensemble
        print("  ✅ Ensemble trained")
    except Exception as e:
        print(f"  ❌ Ensemble failed: {e}")
else:
    print("\n⚠️ Not enough models for ensemble. Need at least 2 models.")

# =============================================================================
# STEP 4: Save Models
# =============================================================================

print("\nStep 4: Saving models...")

os.makedirs(paths.models_dir, exist_ok=True)

for name, model in models.items():
    try:
        with open(f"{paths.models_dir}/model_{name}.pkl", 'wb') as f:
            pickle.dump(model, f)
        print(f"  ✅ Saved {name} model")
    except Exception as e:
        print(f"  ❌ Could not save {name}: {e}")

# =============================================================================
# STEP 5: Summary
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING COMPLETED!")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
for name in models.keys():
    print(f"  - {name}")
print(f"\nModels saved to: {paths.models_dir}")


MODULE 8: MODEL TRAINING

Step 1: Ensuring data is numeric...
Pre-processing data for training...
Loading data from parquet...
Training data shape: (3528464, 121)
Training data types: {dtype('float64'): 64, dtype('int64'): 36, dtype('int32'): 21}

Step 2: Loading tuned parameters...
✅ Loaded tuned parameters
  logistic: ['C', 'solver', 'class_weight']
  random_forest: ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'max_features', 'class_weight']
  xgboost: ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'scale_pos_weight', 'tree_method']
  lightgbm: ['n_estimators', 'num_leaves', 'max_depth', 'learning_rate', 'feature_fraction', 'bagging_fraction', 'bagging_freq', 'is_unbalance']
  catboost: ['iterations', 'depth', 'learning_rate', 'l2_leaf_reg', 'border_count', 'auto_class_weights']

Step 3: Training models...

Training Logistic Regression...
  ❌ Logistic Regression failed: Unable to allocate 969. MiB for an array with shape (36

In [21]:
# =============================================================================
# DIAGNOSTIC: DATA QUALITY CHECK (Using PySpark)
# =============================================================================

print("\n" + "=" * 80)
print("DIAGNOSTIC: DATA QUALITY CHECK")
print("=" * 80)

# Check target distribution using Spark
target_dist = train_df.groupBy("target").count().toPandas()
print(f"\nTarget Distribution:")
for _, row in target_dist.iterrows():
    target_label = "Default (1)" if row['target'] == 1 else "Non-Default (0)"
    pct = row['count'] / target_dist['count'].sum() * 100
    print(f"  {target_label}: {row['count']:,} ({pct:.2f}%)")

# Check for leakage columns
leakage_cols = ['ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 
                'ACTUAL_LOSS_CALCULATION', 'DELINQUENT_ACCRUED_INTEREST',
                'ZERO_BALANCE_REMOVAL_UPB', 'MI_RECOVERIES', 'NET_SALE_PROCEEDS',
                'NON_MI_RECOVERIES', 'TOTAL_EXPENSES', 'LEGAL_COSTS',
                'MAINTENANCE_AND_PRESERVATION_COSTS', 'TAXES_AND_INSURANCE',
                'MISCELLANEOUS_EXPENSES', 'CURRENT_MONTH_MODIFICATION_COST',
                'CUMULATIVE_MODIFICATION_COST', 'DDLPI', 'DEFECT_SETTLEMENT_DATE']

present_leakage = [col for col in leakage_cols if col in train_df.columns]
if present_leakage:
    print(f"\n⚠️ WARNING: Potential data leakage columns found: {present_leakage}")
    print("These should be removed from features!")

# =============================================================================
# FIX 1: Remove Data Leakage Columns using PySpark
# =============================================================================

print("\n" + "=" * 80)
print("FIX 1: Removing Data Leakage Columns using PySpark")
print("=" * 80)

# Columns to drop
drop_cols = [col for col in leakage_cols if col in train_df.columns]
print(f"Removing {len(drop_cols)} leakage columns")

# Drop columns from Spark DataFrames
train_df_clean = train_df.drop(*drop_cols) if drop_cols else train_df
val_df_clean = val_df.drop(*drop_cols) if drop_cols and val_df is not None else val_df
test_df_clean = test_df.drop(*drop_cols) if drop_cols else test_df

print(f"Training data shape after cleaning: {train_df_clean.count():,} rows, {len(train_df_clean.columns)} columns")

# =============================================================================
# FIX 2: Sample Data for Modeling (20% of data)
# =============================================================================

print("\n" + "=" * 80)
print("FIX 2: Sampling Data for Modeling")
print("=" * 80)

# Use 20% of data for training (manageable for pandas)
sample_fraction = 0.20
print(f"Sampling {sample_fraction*100:.0f}% of data for modeling...")

# Sample data
train_sample = train_df_clean.sample(fraction=sample_fraction, seed=42)
print(f"Sampled {train_sample.count():,} training records")

if val_df_clean is not None:
    val_sample = val_df_clean.sample(fraction=sample_fraction, seed=42)
    print(f"Sampled {val_sample.count():,} validation records")
else:
    val_sample = None

test_sample = test_df_clean.sample(fraction=sample_fraction, seed=42)
print(f"Sampled {test_sample.count():,} test records")

# =============================================================================
# FIX 3: Convert to Pandas with Memory Optimization
# =============================================================================

print("\n" + "=" * 80)
print("FIX 3: Converting to Pandas with Memory Optimization")
print("=" * 80)

def to_pandas_optimized(spark_df):
    """Convert Spark DataFrame to Pandas with memory optimization."""
    # First, cache the data
    spark_df.cache()
    
    # Convert to pandas
    pdf = spark_df.toPandas()
    
    # Optimize memory usage
    for col in pdf.columns:
        # Convert object columns to category
        if pdf[col].dtype == 'object':
            unique_count = pdf[col].nunique()
            total_count = len(pdf)
            if unique_count < total_count * 0.1:  # If less than 10% unique
                pdf[col] = pdf[col].astype('category')
        
        # Downcast numeric columns
        if pd.api.types.is_numeric_dtype(pdf[col]):
            pdf[col] = pd.to_numeric(pdf[col], downcast='float')
    
    return pdf

print("Converting training data...")
X_train_pd = to_pandas_optimized(train_sample.drop('target'))
y_train_pd = train_sample.select('target').toPandas()['target']

if val_sample is not None:
    print("Converting validation data...")
    X_val_pd = to_pandas_optimized(val_sample.drop('target'))
    y_val_pd = val_sample.select('target').toPandas()['target']
else:
    X_val_pd = None
    y_val_pd = None

print("Converting test data...")
X_test_pd = to_pandas_optimized(test_sample.drop('target'))
y_test_pd = test_sample.select('target').toPandas()['target']

print(f"Training data shape: {X_train_pd.shape}")
print(f"Training data memory: {X_train_pd.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Test data shape: {X_test_pd.shape}")

# =============================================================================
# FIX 4: Enhanced Data Preprocessing (Memory Efficient)
# =============================================================================

print("\n" + "=" * 80)
print("FIX 4: Enhanced Data Preprocessing")
print("=" * 80)

def enhanced_preprocess_memory_efficient(X_train, X_val=None, X_test=None):
    """
    Memory-efficient preprocessing with proper handling of all data types.
    """
    from sklearn.preprocessing import LabelEncoder
    import pandas as pd
    import numpy as np
    
    # Work with copies
    X_train_enc = X_train.copy()
    X_val_enc = X_val.copy() if X_val is not None else None
    X_test_enc = X_test.copy() if X_test is not None else None
    
    # Identify categorical columns
    cat_cols = X_train_enc.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Also identify columns with few unique values
    for col in X_train_enc.columns:
        if col not in cat_cols:
            unique_count = X_train_enc[col].nunique()
            if unique_count < 20 and X_train_enc[col].dtype in ['int64', 'float64']:
                cat_cols.append(col)
    
    print(f"Found {len(cat_cols)} categorical columns")
    
    # Encode categorical columns
    for col in cat_cols:
        le = LabelEncoder()
        
        # Combine all data for fitting
        if X_val_enc is not None:
            combined = pd.concat([X_train_enc[col], X_val_enc[col]], axis=0)
        else:
            combined = X_train_enc[col]
        
        # Fit encoder
        combined_clean = combined.fillna('MISSING').astype(str)
        combined_clean = combined_clean.replace('nan', 'MISSING')
        combined_clean = combined_clean.replace('None', 'MISSING')
        le.fit(combined_clean)
        
        # Transform each dataset
        for df in [X_train_enc, X_val_enc, X_test_enc]:
            if df is not None and col in df.columns:
                df[col] = df[col].fillna('MISSING').astype(str)
                df[col] = df[col].replace('nan', 'MISSING')
                df[col] = df[col].replace('None', 'MISSING')
                df[col] = le.transform(df[col])
    
    # Convert to numeric
    for df in [X_train_enc, X_val_enc, X_test_enc]:
        if df is not None:
            for col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            df.fillna(0, inplace=True)
            
            # Downcast to save memory
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    df[col] = pd.to_numeric(df[col], downcast='float')
    
    return X_train_enc, X_val_enc, X_test_enc, cat_cols

# Apply preprocessing
print("\nPreprocessing data...")
X_train_processed, X_val_processed, X_test_processed, cat_cols = enhanced_preprocess_memory_efficient(
    X_train_pd, X_val_pd, X_test_pd
)

print(f"Processed training data shape: {X_train_processed.shape}")
print(f"Processed training data memory: {X_train_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# =============================================================================
# FIX 5: Retrain Models with Cleaned Data
# =============================================================================

print("\n" + "=" * 80)
print("FIX 5: Retraining Models with Cleaned Data")
print("=" * 80)

# Reload tuned parameters
import json
if os.path.exists(f"{paths.results_dir}/tuned_params.json"):
    with open(f"{paths.results_dir}/tuned_params.json", 'r') as f:
        tuned_params = json.load(f)
else:
    tuned_params = {}

# Reset models
models = {}

# 1. Logistic Regression
print("\nTraining Logistic Regression...")
try:
    lr = LogisticRegressionModel(
        random_state=model_config.random_state,
        **tuned_params.get('logistic', {})
    )
    lr.fit(X_train_processed, y_train_pd)
    models['logistic'] = lr
    print("  ✅ Logistic Regression trained")
except Exception as e:
    print(f"  ❌ Logistic Regression failed: {e}")

# 2. Random Forest
print("\nTraining Random Forest...")
try:
    rf = RandomForestModel(
        random_state=model_config.random_state,
        **tuned_params.get('random_forest', {})
    )
    rf.fit(X_train_processed, y_train_pd)
    models['random_forest'] = rf
    print("  ✅ Random Forest trained")
except Exception as e:
    print(f"  ❌ Random Forest failed: {e}")

# 3. XGBoost
print("\nTraining XGBoost...")
try:
    import xgboost
    X_train_num = X_train_processed.astype(float)
    X_val_num = X_val_processed.astype(float) if X_val_processed is not None else None
    
    xgb = XGBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get('xgboost', {})
    )
    xgb.fit(X_train_num, y_train_pd, X_val_num, y_val_pd)
    models['xgboost'] = xgb
    print("  ✅ XGBoost trained")
except Exception as e:
    print(f"  ❌ XGBoost failed: {e}")

# 4. LightGBM
print("\nTraining LightGBM...")
try:
    import lightgbm
    lgb = LightGBMModel(
        random_state=model_config.random_state,
        **tuned_params.get('lightgbm', {})
    )
    lgb.fit(X_train_processed, y_train_pd, X_val_processed, y_val_pd)
    models['lightgbm'] = lgb
    print("  ✅ LightGBM trained")
except Exception as e:
    print(f"  ❌ LightGBM failed: {e}")

# 5. CatBoost (with categorical columns from original data)
print("\nTraining CatBoost...")
try:
    import catboost
    # Use original data for CatBoost (it handles categorical internally)
    cat_cols_original = X_train_pd.select_dtypes(include=['object', 'category']).columns.tolist()
    
    cat = CatBoostModel(
        random_state=model_config.random_state,
        cat_features=cat_cols_original,
        **tuned_params.get('catboost', {})
    )
    cat.fit(X_train_pd, y_train_pd, X_val_pd, y_val_pd)
    models['catboost'] = cat
    print("  ✅ CatBoost trained")
except Exception as e:
    print(f"  ❌ CatBoost failed: {e}")

# 6. Ensemble
if len(models) >= 2:
    print("\nTraining Stacking Ensemble...")
    try:
        ensemble = StackingEnsemble(
            base_models=list(models.values()),
            meta_model=LightGBMModel(
                random_state=model_config.random_state,
                is_unbalance=True
            ) if 'lightgbm' in models else LogisticRegressionModel(
                random_state=model_config.random_state
            ),
            cv_folds=min(3, model_config.cv_folds),
            random_state=model_config.random_state
        )
        ensemble.fit(X_train_processed, y_train_pd, X_val_processed, y_val_pd)
        models['ensemble'] = ensemble
        print("  ✅ Ensemble trained")
    except Exception as e:
        print(f"  ❌ Ensemble failed: {e}")

# Save models
print("\nSaving models...")
for name, model in models.items():
    with open(f"{paths.models_dir}/model_{name}_clean.pkl", 'wb') as f:
        pickle.dump(model, f)
    print(f"  Saved {name}_clean model")

# =============================================================================
# FIX 6: Re-evaluate Models
# =============================================================================

print("\n" + "=" * 80)
print("FIX 6: Re-evaluating Models")
print("=" * 80)

from evaluation.metrics import CreditRiskMetrics

# Prepare test data
X_test_sklearn = X_test_processed
X_test_xgboost = X_test_processed.astype(float)
X_test_catboost = X_test_pd

model_data_map = {
    'logistic': X_test_sklearn,
    'random_forest': X_test_sklearn,
    'xgboost': X_test_xgboost,
    'lightgbm': X_test_sklearn,
    'catboost': X_test_catboost,
    'ensemble': X_test_sklearn
}

results = {}
metrics_calculator = CreditRiskMetrics()

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    try:
        X_test_model = model_data_map.get(name, X_test_sklearn)
        
        # Get predictions
        y_proba = model.predict_proba(X_test_model)[:, 1]
        y_pred = model.predict(X_test_model)
        
        # Compute metrics
        metrics = metrics_calculator.evaluate(y_test_pd, y_pred, y_proba)
        metrics['name'] = name
        metrics['y_proba'] = y_proba
        metrics['y_pred'] = y_pred
        
        results[name] = metrics
        
        print(f"  ✅ ROC-AUC: {metrics['roc_auc']:.4f}")
        print(f"  ✅ PR-AUC: {metrics['pr_auc']:.4f}")
        print(f"  ✅ F1: {metrics['f1']:.4f}")
        print(f"  ✅ KS: {metrics['ks_statistic']:.4f}")
        
    except Exception as e:
        print(f"  ❌ Evaluation failed: {e}")
        import traceback
        traceback.print_exc()

# =============================================================================
# RESULTS SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("FINAL RESULTS SUMMARY")
print("=" * 80)

if results:
    results_df = pd.DataFrame([
        {
            'model': name,
            'roc_auc': metrics['roc_auc'],
            'pr_auc': metrics['pr_auc'],
            'f1': metrics['f1'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'balanced_accuracy': metrics['balanced_accuracy'],
            'mcc': metrics['mcc'],
            'brier_score': metrics['brier_score'],
            'log_loss': metrics['log_loss'],
            'ks_statistic': metrics['ks_statistic']
        }
        for name, metrics in results.items()
    ])
    results_df.to_csv(f"{paths.results_dir}/model_results_clean.csv", index=False)
    
    print("\nMODEL PERFORMANCE SUMMARY:")
    print("-" * 80)
    print(f"{'Model':<20} {'ROC-AUC':<10} {'PR-AUC':<10} {'F1':<10} {'KS':<10}")
    print("-" * 80)
    for _, row in results_df.iterrows():
        print(f"{row['model']:<20} {row['roc_auc']:<10.4f} {row['pr_auc']:<10.4f} "
              f"{row['f1']:<10.4f} {row['ks_statistic']:<10.4f}")
    print("-" * 80)
else:
    print("⚠️ No results to display")

print(f"\nResults saved to: {paths.results_dir}/model_results_clean.csv")


DIAGNOSTIC: DATA QUALITY CHECK

Target Distribution:
  Non-Default (0): 3,474,779 (98.48%)
  Default (1): 53,685 (1.52%)

⚠️ WARNING: Potential data leakage columns found: ['ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'ACTUAL_LOSS_CALCULATION', 'DELINQUENT_ACCRUED_INTEREST', 'ZERO_BALANCE_REMOVAL_UPB', 'MI_RECOVERIES', 'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES', 'TOTAL_EXPENSES', 'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS', 'TAXES_AND_INSURANCE', 'MISCELLANEOUS_EXPENSES', 'CURRENT_MONTH_MODIFICATION_COST', 'CUMULATIVE_MODIFICATION_COST', 'DDLPI', 'DEFECT_SETTLEMENT_DATE']
These should be removed from features!

FIX 1: Removing Data Leakage Columns using PySpark
Removing 17 leakage columns
Training data shape after cleaning: 3,528,464 rows, 105 columns

FIX 2: Sampling Data for Modeling
Sampling 20% of data for modeling...
Sampled 706,246 training records
Sampled 177,670 validation records
Sampled 448,792 test records

FIX 3: Converting to Pandas with Memory Optimization
C

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:

# =============================================================================
# MODULE 10: PROBABILITY CALIBRATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 10: PROBABILITY CALIBRATION")
print("=" * 80)

from evaluation.calibration import ProbabilityCalibrator

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING CALIBRATED MODELS (Skip Processing)
# -----------------------------------------------------------------------------
"""
# LOAD EXISTING CALIBRATED MODELS
print("\nLoading existing calibrated models...")
calibrated_models = {}
model_names = ['logistic_calibrated', 'random_forest_calibrated', 'xgboost_calibrated',
               'lightgbm_calibrated', 'catboost_calibrated', 'ensemble_calibrated']
for name in model_names:
    try:
        with open(f"{paths.models_dir}/model_{name}.pkl", 'rb') as f:
            calibrated_models[name] = pickle.load(f)
        print(f"Loaded {name}")
    except Exception as e:
        print(f"Could not load {name}: {e}")
"""

# -----------------------------------------------------------------------------
# OPTION B: RUN CALIBRATION FROM SCRATCH
# -----------------------------------------------------------------------------

# RUN CALIBRATION FROM SCRATCH
print("\nRunning probability calibration...")

# Ensure data is loaded
if 'X_train' not in dir():
    pkl_dir = f"{paths.features_dir}/pandas"
    X_train = pd.read_pickle(f"{pkl_dir}/X_train.pkl")
    y_train = pd.read_pickle(f"{pkl_dir}/y_train.pkl")
    X_val = pd.read_pickle(f"{pkl_dir}/X_val.pkl")
    y_val = pd.read_pickle(f"{pkl_dir}/y_val.pkl")

# Ensure models are loaded
if 'models' not in dir() or not models:
    model_names = ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']
    models = {}
    for name in model_names:
        try:
            with open(f"{paths.models_dir}/model_{name}.pkl", 'rb') as f:
                models[name] = pickle.load(f)
            print(f"Loaded {name} model")
        except Exception as e:
            print(f"Could not load {name} model: {e}")

calibrated_models = {}

for name, model in models.items():
    print(f"\nCalibrating {name}...")
    
    calibrator = ProbabilityCalibrator(method='isotonic')
    calibrated_model = calibrator.calibrate(
        model, X_train, y_train, X_val, y_val
    )
    calibrated_models[f"{name}_calibrated"] = calibrated_model

# Save calibrated models
for name, model in calibrated_models.items():
    with open(f"{paths.models_dir}/model_{name}.pkl", 'wb') as f:
        pickle.dump(model, f)
    print(f"Saved {name}")

print("\nCalibration completed!")



In [ ]:

# =============================================================================
# MODULE 11: CREDIT SCORE GENERATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 11: CREDIT SCORE GENERATION")
print("=" * 80)

from scoring.score_generator import CreditScoreGenerator

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING SCORES (Skip Processing)
# -----------------------------------------------------------------------------
"""
# LOAD EXISTING SCORES
print("\nLoading existing credit scores...")
score_results = pd.read_csv(f"{paths.results_dir}/scores.csv")
print(f"Loaded {len(score_results):,} scores")
print("\nScore Distribution:")
print(score_results['credit_score'].describe())
"""

# -----------------------------------------------------------------------------
# OPTION B: GENERATE SCORES FROM SCRATCH
# -----------------------------------------------------------------------------

# GENERATE SCORES FROM SCRATCH
print("\nGenerating credit scores...")

# Ensure data is loaded
if 'X_test' not in dir():
    pkl_dir = f"{paths.features_dir}/pandas"
    X_test = pd.read_pickle(f"{pkl_dir}/X_test.pkl")
    y_test = pd.read_pickle(f"{pkl_dir}/y_test.pkl")

# Load ensemble model
try:
    with open(f"{paths.models_dir}/model_ensemble.pkl", 'rb') as f:
        ensemble = pickle.load(f)
    print("Loaded ensemble model")
except:
    with open(f"{paths.models_dir}/model_lightgbm.pkl", 'rb') as f:
        ensemble = pickle.load(f)
    print("Loaded LightGBM model (ensemble not available)")

# Get probabilities
y_proba = ensemble.predict_proba(X_test)[:, 1]

# Create score generator
score_generator = CreditScoreGenerator(
    min_score=300,
    max_score=900,
    target_default_rate=0.05,
    pdo=20
)

# Generate scores
score_results = pd.DataFrame({
    'probability': y_proba,
    'true_label': y_test
})

score_results = score_generator.generate_all_scores(
    score_results,
    prob_col='probability',
    score_col='credit_score'
)

# Score distribution
print("\nScore Distribution:")
print(score_results['credit_score'].describe())

# Risk band analysis
print("\nRisk Band Analysis:")
for band in ['Low', 'Medium', 'High']:
    band_data = score_results[score_results['risk_band'] == band]
    if len(band_data) > 0:
        default_rate = band_data['true_label'].mean()
        print(f"  {band}:")
        print(f"    Observations: {len(band_data):,}")
        print(f"    Default Rate: {default_rate:.2%}")
        print(f"    % of Portfolio: {len(band_data)/len(score_results):.2%}")

# Save score results
score_results.to_csv(f"{paths.results_dir}/scores.csv", index=False)
print(f"\nScores saved to {paths.results_dir}/scores.csv")



In [ ]:

# =============================================================================
# MODULE 12: VISUALIZATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 12: VISUALIZATION")
print("=" * 80)

from evaluation.plots import CreditRiskVisualizer

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING VISUALIZATIONS (Skip Processing)
# -----------------------------------------------------------------------------
"""
# LOAD EXISTING VISUALIZATIONS
print("\nExisting visualizations are in:")
print(f"  {paths.results_dir}/plots/")
import os
if os.path.exists(f"{paths.results_dir}/plots"):
    for f in os.listdir(f"{paths.results_dir}/plots"):
        print(f"    {f}")
"""

# -----------------------------------------------------------------------------
# OPTION B: CREATE VISUALIZATIONS FROM SCRATCH
# -----------------------------------------------------------------------------

# CREATE VISUALIZATIONS FROM SCRATCH
print("\nCreating visualizations...")

# Ensure data is loaded
if 'X_test' not in dir():
    pkl_dir = f"{paths.features_dir}/pandas"
    X_test = pd.read_pickle(f"{pkl_dir}/X_test.pkl")
    y_test = pd.read_pickle(f"{pkl_dir}/y_test.pkl")

# Ensure results are available
if 'results' not in dir() or not results:
    # Load evaluation results
    results_df = pd.read_csv(f"{paths.results_dir}/model_results.csv")
    results = {}
    for _, row in results_df.iterrows():
        name = row['model']
        # Load predictions if available
        try:
            preds = pd.read_csv(f"{paths.results_dir}/predictions_{name}.csv")
            results[name] = {
                'name': name,
                'roc_auc': row['roc_auc'],
                'pr_auc': row['pr_auc'],
                'f1': row['f1'],
                'precision': row['precision'],
                'recall': row['recall'],
                'balanced_accuracy': row['balanced_accuracy'],
                'mcc': row['mcc'],
                'brier_score': row['brier_score'],
                'log_loss': row['log_loss'],
                'ks_statistic': row['ks_statistic'],
                'y_proba': preds['y_proba'].values,
                'y_pred': preds['y_pred'].values
            }
        except:
            # Use sample predictions if not available
            model_name = f"model_{name}.pkl"
            try:
                with open(f"{paths.models_dir}/{model_name}", 'rb') as f:
                    model = pickle.load(f)
                y_proba = model.predict_proba(X_test)[:, 1]
                y_pred = model.predict(X_test)
                results[name] = {
                    'name': name,
                    'roc_auc': row['roc_auc'],
                    'pr_auc': row['pr_auc'],
                    'f1': row['f1'],
                    'precision': row['precision'],
                    'recall': row['recall'],
                    'balanced_accuracy': row['balanced_accuracy'],
                    'mcc': row['mcc'],
                    'brier_score': row['brier_score'],
                    'log_loss': row['log_loss'],
                    'ks_statistic': row['ks_statistic'],
                    'y_proba': y_proba,
                    'y_pred': y_pred
                }
            except:
                pass

visualizer = CreditRiskVisualizer(save_dir=f"{paths.results_dir}/plots")

# Create individual model evaluation reports
for name, metrics in results.items():
    if 'y_proba' in metrics and 'feature_importance' in metrics:
        visualizer.create_model_evaluation_report(
            y_test,
            metrics['y_proba'],
            metrics['y_pred'],
            name,
            metrics.get('feature_importance'),
            save_name=f"{name}_report"
        )

# Create ensemble comparison plot
visualizer.create_ensemble_comparison_plot(
    results,
    save_name="ensemble_comparison"
)

print(f"\nVisualizations saved to {paths.results_dir}/plots/")



In [ ]:

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def check_data_exists():
    """Check which data files exist."""
    files = {
        'Bronze Origination': paths.origination_bronze,
        'Bronze Performance': paths.performance_bronze,
        'Silver Origination': f"{paths.silver_dir}/origination_cleaned.parquet",
        'Silver Performance': f"{paths.silver_dir}/performance_cleaned.parquet",
        'Features': paths.feature_dataset,
        'Dataset with Target': f"{paths.features_dir}/dataset_with_target.parquet",
        'Train Data': paths.train_data,
        'Validation Data': paths.val_data,
        'Test Data': paths.test_data,
        'Tuned Parameters': f"{paths.results_dir}/tuned_params.json",
        'Models': paths.models_dir,
        'Results': f"{paths.results_dir}/model_results.csv",
        'Scores': f"{paths.results_dir}/scores.csv"
    }
    
    print("\n" + "=" * 80)
    print("DATA AVAILABILITY CHECK")
    print("=" * 80)
    
    for name, path in files.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"{status} {name}: {path}")
    
    # Check model files
    model_names = ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']
    print("\nModels:")
    for name in model_names:
        path = f"{paths.models_dir}/model_{name}.pkl"
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"{status} {name}")

def print_module_usage():
    """Print usage instructions for this notebook."""
    print("\n" + "=" * 80)
    print("MODULE USAGE INSTRUCTIONS")
    print("=" * 80)
    print("""
    To run a specific module:
    1. Uncomment the OPTION A (LOAD) section to use existing data
    2. OR uncomment the OPTION B (PROCESS) section to process from scratch
    3. Comment out the other option
    4. Run the cell
    
    Recommended workflow:
    1. First time: Run all modules with OPTION B (PROCESS)
    2. Subsequent runs: Use OPTION A (LOAD) for modules you don't want to reprocess
    
    To check what data exists:
    check_data_exists()
    
    Module Dependencies:
    - Module 1 (Ingestion) -> Bronze data
    - Module 2 (Cleaning) -> Silver data
    - Module 3 (Feature Engineering) -> Features
    - Module 4 (Target Creation) -> Dataset with target
    - Module 5 (Dataset Creation & Split) -> Train/Val/Test splits
    - Module 6 (Pandas Data) -> Pandas DataFrames
    - Module 7 (Tuning) -> Tuned parameters
    - Module 8 (Training) -> Trained models
    - Module 9 (Evaluation) -> Evaluation results
    - Module 10 (Calibration) -> Calibrated models
    - Module 11 (Scoring) -> Credit scores
    - Module 12 (Visualization) -> Plots
    """)

# Run this to check what data exists
check_data_exists()

# Run this to see usage instructions
# print_module_usage()

print("\n" + "=" * 80)
print("NOTEBOOK READY")
print("=" * 80)
print("""
To use this notebook:
1. Check what data exists: check_data_exists()
2. Uncomment the desired option (LOAD or PROCESS) for each module
3. Run each cell sequentially
4. Use LOAD options for modules where you already have processed data
5. Use PROCESS options for modules you want to run from scratch
""")


DATA AVAILABILITY CHECK
✅ Bronze Origination: D:/Projects/credit_risk_scoring/data/bronze/origination_bronze.parquet
✅ Bronze Performance: D:/Projects/credit_risk_scoring/data/bronze/performance_bronze.parquet
✅ Silver Origination: D:/Projects/credit_risk_scoring/data/silver/origination_cleaned.parquet
✅ Silver Performance: D:/Projects/credit_risk_scoring/data/silver/performance_cleaned.parquet
✅ Features: D:/Projects/credit_risk_scoring/data/features/feature_dataset.parquet
✅ Dataset with Target: D:/Projects/credit_risk_scoring/data/features/dataset_with_target.parquet
✅ Train Data: D:/Projects/credit_risk_scoring/data/features/train_data.parquet
✅ Validation Data: D:/Projects/credit_risk_scoring/data/features/val_data.parquet
✅ Test Data: D:/Projects/credit_risk_scoring/data/features/test_data.parquet
❌ Tuned Parameters: D:/Projects/credit_risk_scoring/results/tuned_params.json
✅ Models: D:/Projects/credit_risk_scoring/models
❌ Results: D:/Projects/credit_risk_scoring/results/model_